# DataPy · Übungsset für die Wiederholung von Test 2

12 Aufgaben im Stil des Tests: Funktion schreiben, Testzelle darunter ausführen. Lösungen stehen im Notebook `Uebungsset_Test2_Loesungen.ipynb`.

**Reihenfolge**
1. Setup-Zelle ausführen.
2. W1 bis B3 mit dem Werkzeugkasten lösen. Bei jeder Aufgabe zuerst das Grundgerüst als Kommentare hinschreiben (einlesen, bereinigen, …), dann Zeile für Zeile füllen.
3. M1 und M2 am Ende **mit Timer und ohne Werkzeugkasten**.
4. Was nicht auf Anhieb klappte, am nächsten Tag aus einer leeren Zelle wiederholen.

In [213]:
import os
import numpy as np
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

TMP = "uebung_tmp"
os.makedirs(TMP, exist_ok=True)

def write(name, text):
    """Hilfsfunktion für die Tests: schreibt eine Textdatei in den Übungsordner."""
    path = os.path.join(TMP, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

def out(name):
    return os.path.join(TMP, name)


## Werkzeugkasten

Diese Bausteine reichen für alle Aufgaben. Die mit ★ stehen **nicht** auf den Folien, kamen aber im Test oder in den Übungen vor.

**Grundgerüst fast jeder pandas-Aufgabe:** einlesen → bereinigen → berechnen → Ausgabe formen → schreiben / zurückgeben

| Schritt | Baustein |
|---|---|
| einlesen | `pd.read_csv(path)` |
| Duplikate ★ | `df.drop_duplicates()` |
| fehlende Werte | `df.dropna()` · ★ `df.dropna(subset=['temp'])` (nur diese Spalte prüfen) · `s.fillna(0)` · `s.isna().sum()` ★ |
| filtern | `df[df['x'] > 3]` · `df[(df.a > 1) & (df.b == 'x')]` |
| neue Spalte | `df['neu'] = df['a'] / df['b']` |
| gruppieren | `df.groupby('k')['x'].mean()` · `.agg(['mean', 'count'])` · ★ `'count'`, `'max'`, `'min'` gehen auch |
| umbenennen | `.rename(columns={'mean': 'mean_temp'})` |
| runden ★ | `s.round(1)` |
| sortieren | `df.sort_values(by='x')` · ★ `ascending=False` · ★ `df.sort_index()` |
| Index → Spalte ★ | `df.reset_index()` (nach groupby!) · `df.reset_index(drop=True)` (Index neu 0, 1, 2 …) |
| Spalten wählen/ordnen | `df[['station', 'mean_temp', 'n']]` |
| verbinden | `pd.merge(a, b, on='key', how='left')` |
| schreiben | `df.to_csv(path)` · ★ `df.to_csv(path, index=False)` |
| zurückgeben | `len(df)` · `int(...)` · `float(...)` · `s.to_dict()` ★ · `s.tolist()` · ★ `s.idxmax()` (Label des Maximums) |
| Kopie ★ | `df.copy()` wenn das Original unverändert bleiben soll |

**NumPy:** `M.sum(axis=0)` je Spalte, `axis=1` je Zeile · `np.nanmean` · `np.isnan(a)` ★ · `a[a > 5]` · `a[mask] = wert` · `np.argmax`
**Broadcasting ★:** Formen von **rechts** vergleichen; passt, wenn gleich oder eine davon 1. `(3, 4)` + `(4,)` ✓ · `(3, 4)` + `(3,)` ✗ → `.reshape((-1, 1))` macht `(3, 1)` ✓

**Beautiful Soup:** `soup.find('h1')` → erstes Element oder ★ `None` · `soup.find_all('li')` → Liste (leer: `[]`) · `tag.get_text().strip()` · `a.get('href')` · `element.find(...)` sucht nur innerhalb · Liste von Dicts → `pd.DataFrame(rows)`


## W1 · Warm-up: Was kommt raus?

Trage für jeden Ausdruck die **Form des Ergebnisses** als Tupel ein, z. B. `(3, 4)`, oder den String `"Error"`, wenn ein `ValueError` entsteht. Erst überlegen, dann die Testzelle ausführen. (Frage 2 im Test war genau dieser Typ.)

```python
a) np.ones((3, 4)) + np.arange(4)
b) np.ones((3, 4)) + np.arange(3)
c) np.ones((3, 4)) + np.arange(3).reshape((-1, 1))
d) np.ones((2, 3)) * np.array([10, 100, 1000])
e) np.ones((2, 3)) / np.ones((2, 3)).sum(axis=1)
f) np.arange(3).reshape((3, 1)) + np.arange(4)
g) np.ones((3, 4)).sum(axis=0)
h) np.ones((3, 4)).mean(axis=1)
```

In [214]:
antworten = {
    "a": (3,4),
    "b": "Error",
    "c": (3,4),
    "d": (2,3),
    "e": "Error",
    "f": (3,4),
    "g": (4,),
    "h": (3,),
}

print(np.zeros((4,1)), "\n", np.ones((1,4)), "\n", np.zeros(4,))

[[0.]
 [0.]
 [0.]
 [0.]] 
 [[1. 1. 1. 1.]] 
 [0. 0. 0. 0.]


In [215]:
_ausdruecke = {
    "a": lambda: np.ones((3, 4)) + np.arange(4),
    "b": lambda: np.ones((3, 4)) + np.arange(3),
    "c": lambda: np.ones((3, 4)) + np.arange(3).reshape((-1, 1)),
    "d": lambda: np.ones((2, 3)) * np.array([10, 100, 1000]),
    "e": lambda: np.ones((2, 3)) / np.ones((2, 3)).sum(axis=1),
    "f": lambda: np.arange(3).reshape((3, 1)) + np.arange(4),
    "g": lambda: np.ones((3, 4)).sum(axis=0),
    "h": lambda: np.ones((3, 4)).mean(axis=1),
}
_falsch = 0
for k, f in _ausdruecke.items():
    try:
        richtig = f().shape
    except ValueError:
        richtig = "Error"
    if antworten[k] == richtig:
        print(f"✓ {k}")
    else:
        _falsch += 1
        print(f"✗ {k}: deine Antwort {antworten[k]}, richtig ist {richtig}")
assert _falsch == 0, f"{_falsch} falsch"
print("✓ W1 bestanden")

✓ a
✓ b
✓ c
✓ d
✓ e
✓ f
✓ g
✓ h
✓ W1 bestanden


## P1 · CSV bereinigen

Schreibe `clean_csv(input_path, output_path)`:

1. CSV einlesen.
2. Doppelte Zeilen entfernen und alle Zeilen, in denen **irgendein** Wert fehlt.
3. Ergebnis ohne DataFrame-Index unter `output_path` speichern.
4. Die Anzahl der verbleibenden Zeilen als `int` zurückgeben.

In [216]:
def clean_csv(input_path, output_path):
    # TODO
    # einlesen
    df = pd.read_csv(input_path)
    # bereinigen
    df = df.drop_duplicates()
    df = df.dropna()
    df.to_csv(output_path, index=False)
    return len(df)
    

In [217]:
p = write("p1.csv", "name,alter\nAda,36\nBen,\nAda,36\nCem,29\n")
n = clean_csv(p, out("p1_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("p1_out.csv"), encoding="utf-8").read().splitlines() == ["name,alter", "Ada,36.0", "Cem,29.0"]
print("✓ P1 bestanden")

✓ P1 bestanden


## P2 · Umsatz je Produkt

Eine CSV-Datei hat die Spalten `produkt`, `datum`, `umsatz`. Einige Umsätze fehlen, Zeilen können doppelt vorkommen. Schreibe `sales_report(input_path, output_path)`:

1. Einlesen, doppelte Zeilen entfernen, Zeilen ohne `umsatz` entfernen.
2. Je Produkt die **Summe** des Umsatzes (auf 2 Nachkommastellen gerundet) und die **Anzahl** der Verkäufe berechnen.
3. Als CSV speichern mit den Spalten `produkt`, `summe`, `anzahl` in dieser Reihenfolge, sortiert nach `summe` **absteigend**, ohne Index.
4. Den Namen des Produkts mit dem höchsten Umsatz als `str` zurückgeben. Bleiben keine Zeilen übrig: nur die Kopfzeile schreiben und `None` zurückgeben.

In [218]:
def sales_report(input_path, output_path):
    # TODO
    df = pd.read_csv(input_path).drop_duplicates().dropna()
    df = df.groupby("produkt")["umsatz"].agg(["sum", "count"]).round(2).reset_index()
    df = df.rename(columns={"sum": "summe", "count": "anzahl"})
    df = df.sort_values(by="summe", ascending=False)
    df.to_csv(output_path, index=False)
    
    print(df)

    if df.shape[0] == 0:
        return None

    return df.loc[df["summe"].idxmax(), "produkt"]



In [219]:
p = write("p2.csv", "produkt,datum,umsatz\nTee,2026-10-01,3.5\nKaffee,2026-10-01,2.8\nKaffee,2026-10-01,2.8\nKaffee,2026-10-02,2.81\nTee,2026-10-02,\nKakao,2026-10-03,4.0\n")
best = sales_report(p, out("p2_out.csv"))
assert best == "Kaffee", best
assert open(out("p2_out.csv"), encoding="utf-8").read().splitlines() == ["produkt,summe,anzahl", "Kaffee,5.61,2", "Kakao,4.0,1", "Tee,3.5,1"], open(out("p2_out.csv")).read()
p = write("p2_leer.csv", "produkt,datum,umsatz\nTee,2026-10-02,\n")
assert sales_report(p, out("p2_leer_out.csv")) is None
assert open(out("p2_leer_out.csv"), encoding="utf-8").read().strip() == "produkt,summe,anzahl"
print("✓ P2 bestanden")

  produkt  summe  anzahl
0  Kaffee   5.61       2
1   Kakao   4.00       1
2     Tee   3.50       1
Empty DataFrame
Columns: [produkt, summe, anzahl]
Index: []
✓ P2 bestanden


## P3 · Zwei Dateien verbinden

`studierende.csv` hat die Spalten `name`, `kurs_id`. `kurse.csv` hat `kurs_id`, `kurs`. Schreibe `join_courses(students_path, courses_path, output_path)`:

1. Beide Dateien einlesen.
2. So verbinden, dass **alle Studierenden** erhalten bleiben, auch ohne passenden Kurs.
3. Fehlende Kursnamen durch den Text `"unbekannt"` ersetzen.
4. Mit den Spalten `name`, `kurs` (nur diese, in dieser Reihenfolge), sortiert nach `name`, ohne Index speichern.
5. Die Anzahl der Studierenden **ohne** passenden Kurs als `int` zurückgeben.

In [220]:
def join_courses(students_path, courses_path, output_path):
    # TODO
    df_s = pd.read_csv(students_path)
    df_c = pd.read_csv(courses_path)
    df = pd.merge(df_s, df_c, on="kurs_id", how="left")
    result = len(df[df["kurs"].isna()])
    df["kurs"] = df["kurs"].fillna("unbekannt")
    df[["name", "kurs"]].sort_values(by="name").to_csv(output_path, index=False)

    print(df)

    return result
    # return len(df[df["kurs"] == "unbekannt"])

In [221]:
s = write("p3_s.csv", "name,kurs_id\nCem,3\nAda,1\nBen,2\nDia,9\n")
c = write("p3_c.csv", "kurs_id,kurs\n1,DataPy\n2,HCI\n")
n = join_courses(s, c, out("p3_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("p3_out.csv"), encoding="utf-8").read().splitlines() == ["name,kurs", "Ada,DataPy", "Ben,HCI", "Cem,unbekannt", "Dia,unbekannt"]
print("✓ P3 bestanden")

  name  kurs_id       kurs
0  Cem        3  unbekannt
1  Ada        1     DataPy
2  Ben        2        HCI
3  Dia        9  unbekannt
✓ P3 bestanden


## P4 · Noten auswerten (ohne Dateien)

Gegeben ist ein DataFrame mit den Spalten `name`, `semester`, `note`. Schreibe `grade_stats(df)`. Die Funktion gibt ein **Dictionary** zurück mit:

- `"bestanden"`: Anzahl der Zeilen mit `note <= 4.0` (`int`)
- `"beste"`: Name der Person mit der besten (kleinsten) Note (`str`)
- `"schnitt"`: Dictionary `{semester: Mittelwert der Note}`, auf 2 Nachkommastellen gerundet

Zeilen ohne Note werden vorher entfernt. Das übergebene `df` darf **nicht** verändert werden.

In [238]:
def grade_stats(df):
    # TODO
    
    _df = df.dropna(subset=["note"]) # dropna returns a new df

    bestanden = len(_df[_df["note"] <= 4.0])
    print("bestanden:", bestanden)

    beste = str(_df.loc[_df["note"].idxmin(), "name"])
    print("beste:", beste)

    schnitt = _df.groupby("semester")["note"].mean().round(2).to_dict()
    print("schnitt:", schnitt)

    return {"bestanden": bestanden, "beste": beste, "schnitt": schnitt}

In [239]:
_df = pd.DataFrame({"name": ["Ada", "Ben", "Cem", "Dia", "Eli"],
                    "semester": [1, 1, 3, 3, 3],
                    "note": [1.3, 5.0, 2.0, np.nan, 2.7]})
_vorher = _df.copy()
r = grade_stats(_df)
assert r["bestanden"] == 3, r
assert r["beste"] == "Ada", r
assert r["schnitt"] == {1: 3.15, 3: 2.35}, r
assert _df.equals(_vorher), "Original wurde verändert"
print("✓ P4 bestanden")

bestanden: 3
beste: Ada
schnitt: {1: 3.15, 3: 2.35}
✓ P4 bestanden


## N1 · Zeilen und Spalten normieren

Schreibe zwei Funktionen, ohne Schleife:

- `center_columns(M)`: zieht von jeder **Spalte** ihren Mittelwert ab (danach ist jeder Spaltenmittelwert 0).
- `scale_rows(M)`: teilt jede **Zeile** durch ihr Maximum (danach ist das Maximum jeder Zeile 1).

Überlege vorher: Bei welcher der beiden brauchst du `reshape((-1, 1))` und warum?

In [244]:
def center_columns(M):
    # TODO
    print("M:\n", M)
    print("M.mean(axis=0):\n", M.mean(axis=0))
    result = M - M.mean(axis=0)
    print ("result:\n", result)

    return result

def scale_rows(M):
    # TODO
    print("M:\n", M)
    print("M.max():\n", M.max(axis=1))
    print("M.max().reshape((-1,1)):\n", M.max(axis=1).reshape((-1,1)))
    result = M / M.max(axis=1).reshape((-1,1))
    print("result:\n", result)


    return result

In [245]:
_M = np.array([[1., 2., 4.], [3., 6., 8.]])
assert np.allclose(center_columns(_M), [[-1, -2, -2], [1, 2, 2]]), "center_columns"
assert np.allclose(scale_rows(_M), [[0.25, 0.5, 1.0], [0.375, 0.75, 1.0]]), "scale_rows"
print("✓ N1 bestanden")

M:
 [[1. 2. 4.]
 [3. 6. 8.]]
M.mean(axis=0):
 [2. 4. 6.]
result:
 [[-1. -2. -2.]
 [ 1.  2.  2.]]
M:
 [[1. 2. 4.]
 [3. 6. 8.]]
M.max():
 [4. 8.]
M.max().reshape((-1,1)):
 [[4.]
 [8.]]
result:
 [[0.25  0.5   1.   ]
 [0.375 0.75  1.   ]]
✓ N1 bestanden


## N2 · Messwerte bereinigen

`data` ist ein 2D-Array: jede Zeile ist ein Sensor, jede Spalte eine Messung. Schreibe `clean_sensors(data, low, high)`:

1. Eine **Kopie** anlegen (das Original bleibt unverändert).
2. Alle Werte kleiner als `low` oder größer als `high` durch `np.nan` ersetzen.
3. Ein Tupel zurückgeben: `(Mittelwert je Sensor ohne NaN, Index des Sensors mit dem höchsten Mittelwert als int)`.

In [226]:
def clean_sensors(data, low, high):
    # TODO
    arr = data.copy()
    print(arr)
    arr[(arr < low) | (arr > high)] = np.nan
    print(arr)
    mittelwert = np.nanmean(arr, axis=1)
    index = int(np.argmax(mittelwert))
    result = (mittelwert, index)
    print("result:\n", result)
    return result

In [227]:
_d = np.array([[20., 21., 999.], [18., -50., 22.], [25., 26., 27.]])
_vorher = _d.copy()
means, best = clean_sensors(_d, 0, 100)
assert np.allclose(means, [20.5, 20.0, 26.0]), means
assert best == 2 and isinstance(best, int)
assert np.array_equal(_d, _vorher), "Original wurde verändert"
print("✓ N2 bestanden")

[[ 20.  21. 999.]
 [ 18. -50.  22.]
 [ 25.  26.  27.]]
[[20. 21. nan]
 [18. nan 22.]
 [25. 26. 27.]]
result:
 (array([20.5, 20. , 26. ]), 2)
✓ N2 bestanden


## B1 · Links in eine CSV-Datei

Schreibe `links_to_csv(html, output_path)`:

1. Alle `<a>`-Tags finden, die ein `href` haben.
2. Daraus einen DataFrame mit den Spalten `text` (Linktext ohne Leerraum am Rand) und `href` bauen.
3. Ohne Index als CSV speichern.
4. Die Anzahl der Links als `int` zurückgeben.

In [228]:
def links_to_csv(html, output_path):
    # TODO
    soup = BeautifulSoup(html, "html.parser")
    print("soup:\n", soup)
    all_a = soup.find_all("a", href=True)
    print("all_a:\n", all_a)
    rows = []
    for a in all_a:
        rows.append({"text" : a.get_text().strip(),"href": a.get("href")})
    df = pd.DataFrame(rows, columns=["text", "href"])
    df.to_csv(output_path, index=False)
    print("len:\n", len(all_a))
    return len(all_a)


In [229]:
_html = '<p><a href="/a"> Eins </a> <a>ohne Ziel</a> <a href="https://bht-berlin.de">BHT</a></p>'
n = links_to_csv(_html, out("b1_out.csv"))
assert n == 2, n
assert open(out("b1_out.csv"), encoding="utf-8").read().splitlines() == ["text,href", "Eins,/a", "BHT,https://bht-berlin.de"]
assert links_to_csv("<p>nichts</p>", out("b1_leer.csv")) == 0
assert open(out("b1_leer.csv"), encoding="utf-8").read().strip() == "text,href"
print("✓ B1 bestanden")

soup:
 <p><a href="/a"> Eins </a> <a>ohne Ziel</a> <a href="https://bht-berlin.de">BHT</a></p>
all_a:
 [<a href="/a"> Eins </a>, <a href="https://bht-berlin.de">BHT</a>]
len:
 2
soup:
 <p>nichts</p>
all_a:
 []
len:
 0
✓ B1 bestanden


## B2 · Seite zusammenfassen

Schreibe `page_summary(html)`. Die Funktion gibt ein Dictionary zurück:

- `"title"`: Text des `<title>`-Tags, oder `None`, wenn es keinen gibt
- `"h1"`: Text der ersten `<h1>`, oder `None`
- `"n_p"`: Anzahl der `<p>`-Tags
- `"items"`: Liste mit den Texten aller `<li>`-Elemente (ohne Leerraum am Rand)

Achtung: `soup.find(...)` liefert `None`, wenn es nichts findet. `None.get_text()` ist ein `AttributeError`.

In [230]:
def page_summary(html):
    # TODO
    soup = BeautifulSoup(html, "html.parser")
    title_tag = soup.find("title")
    title = title_tag.get_text() if title_tag else None

    h1_tag = soup.find("h1")
    h1 = h1_tag.get_text() if h1_tag else None
    
    n_p = len(soup.find_all("p"))
    items = [li.get_text().strip() for li in soup.find_all("li")]

    result = {"title" : title, "h1" : h1, "n_p" : n_p, "items" : items}
    print("result:\n", result)
    return result

In [231]:
_html = "<html><head><title>Mensa</title></head><body><p>Hallo</p><ul><li> Curry </li><li>Suppe</li></ul><p>Tschüss</p></body></html>"
assert page_summary(_html) == {"title": "Mensa", "h1": None, "n_p": 2, "items": ["Curry", "Suppe"]}, page_summary(_html)
assert page_summary("<h1>Nur Überschrift</h1>") == {"title": None, "h1": "Nur Überschrift", "n_p": 0, "items": []}
print("✓ B2 bestanden")

result:
 {'title': 'Mensa', 'h1': None, 'n_p': 2, 'items': ['Curry', 'Suppe']}
result:
 {'title': None, 'h1': 'Nur Überschrift', 'n_p': 0, 'items': []}
✓ B2 bestanden


## B3 · HTML-Tabelle auswerten

Ein HTML-Text enthält eine Tabelle mit den Spalten `Bezirk` und `Einwohner`. Schreibe `biggest_district(html)`:

1. Die erste Tabelle als DataFrame einlesen (`pd.read_html` braucht `StringIO(html)` und liefert eine **Liste**).
2. Den Namen des Bezirks mit den meisten Einwohnern als `str` zurückgeben.

In [232]:
def biggest_district(html):
    # TODO
    
    df = pd.read_html(StringIO(html))[0]
    result = df.loc[df["Einwohner"].idxmax(), "Bezirk"]
    print("result:\n", result)
    return result



In [233]:
_html = "<table><tr><th>Bezirk</th><th>Einwohner</th></tr><tr><td>Mitte</td><td>332100</td></tr><tr><td>Pankow</td><td>368956</td></tr><tr><td>Spandau</td><td>225420</td></tr></table>"
assert biggest_district(_html) == "Pankow"
print("✓ B3 bestanden")

result:
 Pankow
✓ B3 bestanden


## M1 · Probeklausur-Aufgabe 1 (20 Minuten, ohne Werkzeugkasten)

**Stell einen Timer auf 20 Minuten und scrolle nicht nach oben.**

Eine CSV-Datei enthält Prüfungsergebnisse mit den Spalten `name`, `kurs` und `punkte`. Einige Punktwerte fehlen, außerdem können Zeilen doppelt vorkommen. Implementiere `summarize_exams(input_path, output_path)` mit pandas:

1. CSV-Datei einlesen, doppelte Zeilen sowie alle Zeilen ohne `punkte` entfernen.
2. Für jeden Kurs den Mittelwert der Punkte (auf eine Nachkommastelle gerundet), die höchste Punktzahl und die Anzahl der Ergebnisse bestimmen.
3. Als CSV unter `output_path` speichern, mit den Spalten `kurs`, `mean_punkte`, `max_punkte`, `n` in dieser Reihenfolge. Die Kurse müssen alphabetisch sortiert sein. Der Index darf nicht mitgespeichert werden.
4. Die Anzahl der Kurse als `int` zurückgeben.

Sonderfall: Bleiben keine Ergebnisse übrig, wird eine Datei nur mit der Kopfzeile geschrieben und `0` zurückgegeben.

In [234]:
def summarize_exams(input_path, output_path):
    # TODO
    df = pd.read_csv(input_path).drop_duplicates()
    df = df.dropna(subset=["punkte"])

    result = df.groupby("kurs").agg(
        mean_punkte=("punkte", "mean"),
        max_punkte=("punkte", "max"),
        n=("punkte", "count")
    ).reset_index()

    result["mean_punkte"] = result["mean_punkte"].round(1)
    result = result.sort_values("kurs")

    result.to_csv(output_path, index=False)

    
    print("result:\n", result)

    return len(result)




In [235]:
p = write("m1.csv", "name,kurs,punkte\nAda,HCI,40\nBen,DataPy,31\nBen,DataPy,31\nCem,DataPy,36\nDia,HCI,\nEli,DataPy,38\n")
n = summarize_exams(p, out("m1_out.csv"))
assert n == 2 and isinstance(n, int), n
assert open(out("m1_out.csv"), encoding="utf-8").read().splitlines() == ["kurs,mean_punkte,max_punkte,n", "DataPy,35.0,38.0,3", "HCI,40.0,40.0,1"], open(out("m1_out.csv")).read()
p = write("m1_leer.csv", "name,kurs,punkte\nDia,HCI,\n")
assert summarize_exams(p, out("m1_leer_out.csv")) == 0
assert open(out("m1_leer_out.csv"), encoding="utf-8").read().strip() == "kurs,mean_punkte,max_punkte,n"
print("✓ M1 bestanden")

result:
      kurs  mean_punkte  max_punkte  n
0  DataPy         35.0        38.0  3
1     HCI         40.0        40.0  1
result:
 Empty DataFrame
Columns: [kurs, mean_punkte, max_punkte, n]
Index: []
✓ M1 bestanden


## M2 · Probeklausur-Aufgabe 2 (20 Minuten, ohne Werkzeugkasten)

**Timer auf 20 Minuten.** Diesmal mit Beautiful Soup statt CSV-Eingabe.

Ein HTML-Text enthält Produkte in dieser Form:

```html
<div class="produkt"><h2>Tee</h2><span>3,50 €</span></div>
```

Implementiere `products_to_csv(html, output_path)`:

1. Alle `<h2>`-Tags (Produktnamen) und alle `<span>`-Tags (Preise) auslesen. Beide Listen sind gleich lang und in derselben Reihenfolge.
2. Die Preise in `float` umwandeln (`"3,50 €"` → `3.5`).
3. Einen DataFrame mit den Spalten `produkt` und `preis` bauen, nach `preis` aufsteigend sortieren und ohne Index als CSV speichern.
4. Den Durchschnittspreis, auf 2 Nachkommastellen gerundet, als `float` zurückgeben. Gibt es keine Produkte: nur die Kopfzeile schreiben und `0.0` zurückgeben.

In [236]:
def products_to_csv(html, output_path):
    # TODO
    soup = BeautifulSoup(html, "html.parser")
    h2_tags = soup.find_all("h2")
    h2_text = [h2.get_text().strip() for h2 in h2_tags]
    span_tags = soup.find_all("span")
    span_text = [float(span.get_text().replace("€", "").strip().replace(",",".")) for span in span_tags]
    print(span_text)

    df = pd.DataFrame({
        "produkt" : h2_text,
        "preis" : span_text
    }).sort_values("preis", ascending=True)

    df.to_csv(output_path, index=False)

    if len(df) == 0:
        return 0.0

    result = df["preis"].mean().round(2)
    print("result:\n", result)
    return result


In [237]:
_html = '<div class="produkt"><h2>Tee</h2><span>3,50 €</span></div><div class="produkt"><h2> Kaffee </h2><span>2,80 €</span></div><div class="produkt"><h2>Kakao</h2><span>4,05 €</span></div>'
m = products_to_csv(_html, out("m2_out.csv"))
assert m == 3.45 and isinstance(m, float), m
assert open(out("m2_out.csv"), encoding="utf-8").read().splitlines() == ["produkt,preis", "Kaffee,2.8", "Tee,3.5", "Kakao,4.05"]
assert products_to_csv("<p>leer</p>", out("m2_leer.csv")) == 0.0
assert open(out("m2_leer.csv"), encoding="utf-8").read().strip() == "produkt,preis"
print("✓ M2 bestanden")

[3.5, 2.8, 4.05]
result:
 3.45
[]
✓ M2 bestanden
